In [0]:
# =============================================================
# CELDA 1: SETUP — Imports, Credenciales, Config S3
# =============================================================
import json
import boto3
from functools import reduce
from pyspark.sql.functions import (
    col, coalesce, regexp_extract, trim, lit,
    when, length, lower, row_number, desc, current_timestamp
)
from pyspark.sql.window import Window

# Credenciales (fail-fast)
try:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)
except FileNotFoundError:
    raise SystemExit("❌ aws_secrets.json no encontrado.")
except json.JSONDecodeError:
    raise SystemExit("❌ aws_secrets.json inválido.")

AWS_KEY = config["aws_access_key"]
AWS_SECRET = config["aws_secret_key"]
BUCKET = "bronce-scrap-date"

S3_OPTS = {
    "fs.s3a.access.key": AWS_KEY,
    "fs.s3a.secret.key": AWS_SECRET,
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

print("✅ Setup Silver listo.")

In [0]:
# =============================================================
# CELDA 2: FUNCIÓN — Normalización Universal Multi-Fuente
# =============================================================
# Cada portal usa nombres diferentes para el mismo concepto:
#   precio/price, ubicacion/location/city, titulo/title, url/listing_url
# Esta función usa coalesce() para mapear TODAS las variantes sin if/else.

def _safe_extract_number(col_expr, pattern, cast_type="double"):
    """regexp_extract retorna '' cuando no hay match → cast falla.
    Esta helper retorna NULL en vez de '' antes del cast."""
    extracted = regexp_extract(col_expr, pattern, 1)
    return when(length(extracted) > 0, extracted.cast(cast_type))

def normalizar_fuente(fuente):
    """Lee Bronze Delta de una fuente y normaliza a schema Silver unificado."""
    ruta = f"s3a://{BUCKET}/bronze/{fuente}/"
    print(f"📖 {fuente}...")

    try:
        reader = spark.read.format("delta")
        for k, v in S3_OPTS.items():
            reader = reader.option(k, v)
        df = reader.load(ruta)
    except Exception as e:
        print(f"  ⚠️ Saltando {fuente}: {str(e)[:120]}")
        return None

    cols = set(df.columns)
    def safe(name):
        """Retorna col(name) si existe, lit(None) si no — evita errores por schema variable."""
        return col(name) if name in cols else lit(None).cast("string")

    # === MAPEO UNIVERSAL ===
    df_norm = df.select(
        col("id_inmueble").alias("id_original"),

        # Timestamp: preferir fecha del scraper, fallback a metadata Bronze
        coalesce(
            safe("fecha_extraccion"), safe("extracted_at"),
            safe("ingestion_timestamp"), safe("bronze_ingested_at")
        ).cast("timestamp").alias("fecha_extraccion"),

        # Fuente: preferir metadata Bronze, fallback a columna del scraper
        coalesce(safe("source_system"), safe("portal"), safe("source")).alias("fuente"),

        # URL (bancolombia usa listing_url)
        coalesce(safe("url"), safe("listing_url")).alias("url"),

        # Título (bancolombia/mercadolibre usan title)
        coalesce(safe("titulo"), safe("title")).alias("titulo"),

        # Ubicación (bancolombia usa city, mercadolibre usa location)
        coalesce(safe("ubicacion"), safe("location"), safe("city")).alias("ubicacion_raw"),

        # Precio numérico — YA viene limpio (int64) de TODOS los scrapers
        safe("precio_num").cast("long").alias("precio_num"),

        # Área: extraer número con decimales ("36.74 m2" → 36.74, "" → NULL)
        _safe_extract_number(safe("area"), r"(\d+\.?\d*)", "double").alias("area_m2"),

        # Habitaciones: extraer dígitos ("2 Habs." → 2, "" → NULL)
        _safe_extract_number(
            coalesce(safe("habitaciones"), safe("rooms")), r"(\d+)", "int"
        ).alias("habitaciones"),

        # Baños ("baños 1" → 1, "" → NULL)
        _safe_extract_number(safe("banos"), r"(\d+)", "int").alias("banos"),

        # Garajes/Parqueaderos ("garaje 2" → 2, "" → NULL)
        _safe_extract_number(safe("garajes"), r"(\d+)", "int").alias("garajes"),

        # Tipo de inmueble (fincaraiz/metrocuadrado: tipo_inmueble, mercadolibre: property_type)
        coalesce(safe("tipo_inmueble"), safe("property_type")).alias("tipo_inmueble"),

        # Estado (nuevo/usado — ciencuadras y bancolombia: estado_inmueble/status)
        coalesce(safe("estado_inmueble"), safe("status")).alias("estado_inmueble"),

        # Metadata Bronze (trazabilidad MLOps)
        safe("source_file").alias("source_file"),
        safe("batch_id").alias("batch_id"),
    )

    # === FILTROS DE CALIDAD ===
    df_clean = df_norm.filter(
        (col("precio_num").isNotNull()) &
        (col("precio_num") > 20_000_000) &       # Sin arriendos
        (col("precio_num") < 20_000_000_000)      # Sin outliers extremos
    )

    # Limpiar ubicación vacía / "NaN" / muy corta
    df_clean = df_clean.withColumn(
        "ubicacion_raw",
        when(
            col("ubicacion_raw").isNull() |
            (length(trim(col("ubicacion_raw"))) < 3) |
            (lower(col("ubicacion_raw")) == "nan"),
            lit("Sin Ubicación")
        ).otherwise(trim(col("ubicacion_raw")))
    )

    n = df_clean.count()
    print(f"  ✅ {fuente}: {n} registros válidos")
    return df_clean

In [0]:
# =============================================================
# CELDA 3: EJECUCIÓN — Descubrimiento Dinámico + Unión + Dedup
# =============================================================

# Descubrir fuentes dinámicamente desde Bronze (NO hardcoded)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    region_name="us-east-1",
)

response = s3.list_objects_v2(Bucket=BUCKET, Prefix="bronze/", Delimiter="/")
fuentes = [
    p["Prefix"].replace("bronze/", "").strip("/")
    for p in response.get("CommonPrefixes", [])
    if p["Prefix"].strip("/") != "bronze"
]
print(f"🔍 Fuentes en Bronze: {fuentes}")

# Normalizar cada fuente
dfs = []
for f in fuentes:
    df = normalizar_fuente(f)
    if df is not None:
        dfs.append(df)

if not dfs:
    raise SystemExit("❌ No hay datos para procesar en Bronze.")

# Unión (allowMissingColumns maneja columnas que no existen en todas las fuentes)
df_union = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
print(f"\n📊 Total unificado: {df_union.count()} registros")

# Deduplicación: conservar el registro más reciente por (id_original, fuente)
w = Window.partitionBy("id_original", "fuente").orderBy(desc("fecha_extraccion"))

df_silver = (
    df_union
    .withColumn("_rank", row_number().over(w))
    .filter(col("_rank") == 1)
    .drop("_rank")
    .withColumn("silver_processed_at", current_timestamp())
)

print(f"📉 Registros únicos: {df_silver.count()}")
display(df_silver)

In [0]:
# =============================================================
# CELDA 4: GUARDAR — Silver (Delta) + Gold Consumable (Parquet)
# =============================================================

# Silver: tabla maestra de-duplicada
ruta_silver = f"s3a://{BUCKET}/silver/master_inmuebles/"
writer = df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_silver)
print(f"🥈 Silver guardado: {ruta_silver}")

# Gold consumable: snapshot para API (un solo archivo parquet)
ruta_gold = f"s3a://{BUCKET}/gold/app_consumable/"
writer = df_silver.coalesce(1).write.format("parquet").mode("overwrite")
for k, v in S3_OPTS.items():
    writer = writer.option(k, v)
writer.save(ruta_gold)
print(f"🥇 Gold consumable: {ruta_gold}")